# Matmul: PMPP/Ch. 6

Warp divergence (sum reduction demo), loop unrolling, data prefetching, thread granularity CUDA + Triton + Figure 6.16 benchmark

In [ ]:
import cupy as cp
import numpy as np
import torch
import triton
import triton.language as tl
import matplotlib.pyplot as plt
import matplotlib
import time

print(f'GPU: {cp.cuda.Device().name}')

WIDTH = 256
M_cp = cp.random.randn(WIDTH, WIDTH).astype(cp.float32)
N_cp = cp.random.randn(WIDTH, WIDTH).astype(cp.float32)
REF  = cp.asnumpy(M_cp @ N_cp)

M_t = torch.as_tensor(M_cp, device='cuda')
N_t = torch.as_tensor(N_cp, device='cuda')

def gflops(width, ms):
    return 2 * width**3 / (ms * 1e-3) / 1e9

## Loop Unrolling (CUDA)

Each iteration of the inner `k` loop adds address arithmetic on top of the two FLOPs so only 2/3 of instructions do useful work. `#pragma unroll N` tells the compiler to unroll N iterations, eliminating that overhead.

NVCC unrolls small loops automatically nowadays, but it's worth making it explicit.

In [ ]:
def make_unroll_kernel(tile, unroll):
    pragma = f'#pragma unroll {unroll}' if isinstance(unroll, int) else '#pragma unroll'
    name = f'matmul_tile{tile}_unroll{unroll}'
    src = rf'''
extern "C" __global__
void {name}(const float *M, const float *N, float *P, int width) {{
    __shared__ float Mds[{tile}][{tile}];
    __shared__ float Nds[{tile}][{tile}];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * {tile} + ty;
    int col = blockIdx.x * {tile} + tx;
    float sum = 0.0f;
    for (int t = 0; t < width / {tile}; t++) {{
        Mds[ty][tx] = M[row * width + t * {tile} + tx];
        Nds[ty][tx] = N[(t * {tile} + ty) * width + col];
        __syncthreads();
        {pragma}
        for (int k = 0; k < {tile}; k++)
            sum += Mds[ty][k] * Nds[k][tx];
        __syncthreads();
    }}
    if (row < width && col < width) P[row * width + col] = sum;
}}
'''
    return cp.RawKernel(src, name)

# Sanity check
P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
fn = make_unroll_kernel(16, 4)
GRID = WIDTH // 16
fn((GRID, GRID), (16, 16), (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH)))
assert np.allclose(cp.asnumpy(P), REF, atol=1e-2), 'WRONG'
print('Unroll kernel: correct')

## Data Prefetching (CUDA)

The base kernel does two things in one line: global memory read → shared memory store. Splitting them into separate instructions gives the GPU scheduler more independent work to overlap while waiting for DRAM.

```
// Normal: one instruction, thread stalls waiting for DRAM
Mds[ty][tx] = M[row * width + t * TILE + tx];

// Prefetch: issue the DRAM read early into a register,
// do other work, then store to shared memory later
float reg_M = M[...];   // issue read
... compute current tile ...
Mds[ty][tx] = reg_M;    // store when ready
```

In [ ]:
def make_prefetch_kernel(tile, unroll):
    pragma = f'#pragma unroll {unroll}' if isinstance(unroll, int) else '#pragma unroll'
    name = f'matmul_prefetch_tile{tile}_unroll{unroll}'
    src = rf'''
extern "C" __global__
void {name}(const float *M, const float *N, float *P, int width) {{
    __shared__ float Mds[{tile}][{tile}];
    __shared__ float Nds[{tile}][{tile}];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * {tile} + ty;
    int col = blockIdx.x * {tile} + tx;
    float sum = 0.0f;
    // Prefetch first tile into registers
    float reg_M = M[row * width + tx];
    float reg_N = N[ty * width + col];
    for (int t = 0; t < width / {tile}; t++) {{
        // Store prefetched registers into shared memory
        Mds[ty][tx] = reg_M;
        Nds[ty][tx] = reg_N;
        // Prefetch NEXT tile while we sync and compute current
        if (t + 1 < width / {tile}) {{
            reg_M = M[row * width + (t + 1) * {tile} + tx];
            reg_N = N[((t + 1) * {tile} + ty) * width + col];
        }}
        __syncthreads();
        {pragma}
        for (int k = 0; k < {tile}; k++)
            sum += Mds[ty][k] * Nds[k][tx];
        __syncthreads();
    }}
    if (row < width && col < width) P[row * width + col] = sum;
}}
'''
    return cp.RawKernel(src, name)

P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
fn = make_prefetch_kernel(16, 4)
fn((GRID, GRID), (16, 16), (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH)))
assert np.allclose(cp.asnumpy(P), REF, atol=1e-2), 'WRONG'
print('Prefetch kernel: correct')

## Thread Granularity (CUDA)

Each thread computes more than one output element. For `1×2`, a thread handles `P[row][col]` and `P[row][col2]` loading the M tile once and reusing it for both, halving global memory reads for M.

Trade-off: more registers per thread → potentially fewer concurrent blocks per SM.

In [ ]:
def make_granularity_kernel(tile, gran, prefetch, unroll):
    pragma = f'#pragma unroll {unroll}' if isinstance(unroll, int) else '#pragma unroll'
    name = f'matmul_g{gran}_tile{tile}_pre{int(prefetch)}_unroll{unroll}'

    shared_N = '\n'.join([f'    __shared__ float Nds{i}[{tile}][{tile}];' for i in range(gran)])
    sums     = '\n'.join([f'    float sum{i} = 0.0f;' for i in range(gran)])
    col_defs = '\n'.join([f'    int col{i} = (blockIdx.x * {gran} + {i}) * {tile} + tx;' for i in range(gran)])

    if prefetch:
        reg_init = '\n'.join([f'    float reg_N{i} = N[ty * width + col{i}];' for i in range(gran)])
        reg_init = f'    float reg_M = M[row * width + tx];\n' + reg_init
        nds_store = '\n'.join([f'        Nds{i}[ty][tx] = reg_N{i};' for i in range(gran)])
        reg_next  = '\n'.join([f'            reg_N{i} = N[((t+1)*{tile}+ty)*width+col{i}];' for i in range(gran)])
        reg_next  = f'            reg_M = M[row*width+(t+1)*{tile}+tx];\n' + reg_next
        load_block = f'        Mds[ty][tx] = reg_M;\n{nds_store}\n        if (t+1 < width/{tile}) {{\n{reg_next}\n        }}'
    else:
        nds_store  = '\n'.join([f'        Nds{i}[ty][tx] = N[(t*{tile}+ty)*width+col{i}];' for i in range(gran)])
        load_block = f'        Mds[ty][tx] = M[row*width+t*{tile}+tx];\n{nds_store}'

    acc = '\n'.join([f'            sum{i} += Mds[ty][k] * Nds{i}[k][tx];' for i in range(gran)])
    stores = '\n'.join([f'    if (row < width && col{i} < width) P[row*width+col{i}] = sum{i};' for i in range(gran)])

    src = rf'''
extern "C" __global__
void {name}(const float *M, const float *N, float *P, int width) {{
    __shared__ float Mds[{tile}][{tile}];
{shared_N}
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * {tile} + ty;
{col_defs}
{sums}
{'    float reg_M = M[row * width + tx];' if prefetch else ''}
{''.join([chr(10)+f'    float reg_N{i} = N[ty * width + col{i}];' for i in range(gran)]) if prefetch else ''}
    for (int t = 0; t < width / {tile}; t++) {{
{load_block}
        __syncthreads();
        {pragma}
        for (int k = 0; k < {tile}; k++) {{
{acc}
        }}
        __syncthreads();
    }}
{stores}
}}
'''
    return cp.RawKernel(src, name)

# Sanity check 1x2
P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
fn = make_granularity_kernel(tile=16, gran=2, prefetch=False, unroll=1)
fn((WIDTH//(2*16), WIDTH//16), (16, 16), (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH)))
assert np.allclose(cp.asnumpy(P), REF, atol=1e-2), 'WRONG'
print('Granularity 1x2: correct')

P = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
fn = make_granularity_kernel(tile=16, gran=4, prefetch=False, unroll=1)
fn((WIDTH//(4*16), WIDTH//16), (16, 16), (M_cp.ravel(), N_cp.ravel(), P.ravel(), np.int32(WIDTH)))
assert np.allclose(cp.asnumpy(P), REF, atol=1e-2), 'WRONG'
print('Granularity 1x4: correct')

## Triton prefetch + thread granularity

In Triton, prefetching happens automatically (the compiler pipelines memory loads). Thread granularity maps to the `BLOCK_SIZE` a larger block means each program handles more output elements.

In [ ]:
@triton.jit
def matmul_triton_ch6(M_ptr, N_ptr, P_ptr, width,
                      TILE: tl.constexpr, GRAN: tl.constexpr):
    pid_row = tl.program_id(1)
    pid_col = tl.program_id(0)

    row = pid_row * TILE + tl.arange(0, TILE)[:, None]
    # GRAN output tiles per program in the column direction
    col_base = pid_col * TILE * GRAN

    acc = tl.zeros((TILE, TILE * GRAN), dtype=tl.float32)
    col = col_base + tl.arange(0, TILE * GRAN)[None, :]

    for t in range(0, width, TILE):
        k = t + tl.arange(0, TILE)
        m = tl.load(M_ptr + row * width + k[None, :],
                    mask=(row < width) & (k[None, :] < width), other=0.0)
        n = tl.load(N_ptr + k[:, None] * width + col,
                    mask=(k[:, None] < width) & (col < width), other=0.0)
        acc += tl.dot(m, n)

    tl.store(P_ptr + row * width + col, acc,
             mask=(row < width) & (col < width))


TILE, GRAN = 16, 2
P_t = torch.zeros((WIDTH, WIDTH), device='cuda', dtype=torch.float32)
matmul_triton_ch6[(WIDTH//(TILE*GRAN), WIDTH//TILE)](
    M_t, N_t, P_t, WIDTH, TILE=TILE, GRAN=GRAN
)
assert torch.allclose(P_t, torch.tensor(REF, device='cuda'), atol=1e-2), 'WRONG'
print('Triton ch6 (GRAN=2): correct')

## Figure 6.16 Benchmark

Reproducing the measured performance chart from the book: GFLOPS across tile sizes (8×8, 16×16), thread granularity (1×1, 1×2, 1×4), prefetch (normal / prefetch), and unroll level (1, 2, 4, complete).

In [ ]:
RUNS = 20

def bench(fn, grid, block, args):
    fn(grid, block, args)
    cp.cuda.stream.get_current_stream().synchronize()
    t0 = time.time()
    for _ in range(RUNS):
        fn(grid, block, args)
    cp.cuda.stream.get_current_stream().synchronize()
    ms = (time.time() - t0) / RUNS * 1000
    return gflops(WIDTH, ms)

tiles    = [8, 16]
grans    = [1, 2, 4]
prefetch = [False, True]
unrolls  = [1, 2, 4, 'full']

results = {}  # (tile, gran, pre, unroll) -> gflops

P_out = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
args  = (M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH))

for tile in tiles:
    for gran in grans:
        for pre in prefetch:
            for unroll in unrolls:
                key = (tile, gran, pre, unroll)
                grid_x = WIDTH // (gran * tile)
                grid_y = WIDTH // tile
                if grid_x == 0:
                    results[key] = None
                    continue
                try:
                    fn = make_granularity_kernel(tile, gran, pre, unroll)
                    g  = bench(fn, (grid_x, grid_y), (tile, tile), args)
                    results[key] = g
                    print(f'tile={tile} gran=1x{gran} pre={int(pre)} unroll={unroll}: {g:.1f} GFLOPS')
                except Exception as e:
                    results[key] = None
                    print(f'tile={tile} gran=1x{gran} pre={int(pre)} unroll={unroll}: Cannot run')

In [ ]:
unroll_labels  = ['unroll 1', 'unroll 2', 'unroll 4', 'complete unroll']
unroll_keys    = [1, 2, 4, 'full']

# x-axis groups: (tile, gran) pairs
groups = [(8,1),(8,2),(8,4),(16,1),(16,2),(16,4)]
group_labels = ['1×1','1×2','1×4','1×1','1×2','1×4']

n_groups  = len(groups)      # 6
n_pre     = 2                # normal / prefetch
n_unroll  = len(unroll_keys) # 4 bars per sub-group
bar_w     = 0.18
sub_gap   = 0.05             # gap between normal and prefetch within a group
group_gap = 0.4              # gap between groups

fig, ax = plt.subplots(figsize=(14, 5))

group_centers = []
x = 0.0
for gi, (tile, gran) in enumerate(groups):
    sub_centers = []
    for pi, pre in enumerate([False, True]):
        sub_x = x + pi * (n_unroll * bar_w + sub_gap)
        for ui, (uk, color) in enumerate(zip(unroll_keys, unroll_colors)):
            key = (tile, gran, pre, uk)
            val = results.get(key)
            bx  = sub_x + ui * bar_w
            bar = ax.bar(bx, val, width=bar_w, color=color,
                            edgecolor='black', linewidth=0.5,
                            label=unroll_labels[ui] if gi == 0 and pi == 0 else '')
        sub_centers.append(sub_x + (n_unroll * bar_w) / 2)
    group_center = (sub_centers[0] + sub_centers[1]) / 2
    group_centers.append(group_center)
    # sub-labels: normal / prefetch
    ax.text(sub_centers[0], -12, 'normal',  ha='center', va='top', fontsize=7)
    ax.text(sub_centers[1], -12, 'prefetch',ha='center', va='top', fontsize=7)
    x += n_pre * (n_unroll * bar_w + sub_gap) + group_gap

# Group labels (1×1, 1×2, etc.)
for gc, gl in zip(group_centers, group_labels):
    ax.text(gc, -22, gl, ha='center', va='top', fontsize=8, fontweight='bold')

# Tile section labels
ax.text(np.mean([group_centers[0], group_centers[2]]), -32,
        '8×8 tiles', ha='center', va='top', fontsize=9, fontweight='bold')
ax.text(np.mean([group_centers[3], group_centers[5]]), -32,
        '16×16 tiles', ha='center', va='top', fontsize=9, fontweight='bold')

ax.set_ylabel('GFLOPS')
ax.set_title('Measured performance with various techniques  (Figure 6.16)')
ax.set_xticks([])
ax.set_xlim(-0.2, x - group_gap + 0.2)
ax.set_ylim(0, 140)
ax.legend(loc='upper left', fontsize=8)
ax.text(0.98, 0.98, 'Optimizations', transform=ax.transAxes,
        ha='right', va='top', fontsize=8)
plt.subplots_adjust(bottom=0.2)
plt.tight_layout()
plt.savefig('fig6_16.png', dpi=150, bbox_inches='tight')
plt.show()

#### Triton vs Best CUDA Kernel

Best CUDA config from Figure 6.16 (16x16 tile, 1x4 granularity, prefetch, complete unroll) vs Triton with equivalent settings.

In [ ]:
RUNS = 20

# Best CUDA config from Fig 6.16: 16x16 tile, gran=4, prefetch, complete unroll
best_cuda_fn = make_granularity_kernel(tile=16, gran=4, prefetch=True, unroll="full")
best_grid    = (WIDTH // (4 * 16), WIDTH // 16)
P_out = cp.zeros((WIDTH, WIDTH), dtype=cp.float32)
args  = (M_cp.ravel(), N_cp.ravel(), P_out.ravel(), np.int32(WIDTH))

best_cuda_fn(best_grid, (16, 16), args)  # warmup
cp.cuda.stream.get_current_stream().synchronize()
t0 = time.time()
for _ in range(RUNS):
    best_cuda_fn(best_grid, (16, 16), args)
cp.cuda.stream.get_current_stream().synchronize()
t_best_cuda = (time.time() - t0) / RUNS * 1000

# Triton with TILE=16, GRAN=4
P_out_t = torch.zeros((WIDTH, WIDTH), device="cuda", dtype=torch.float32)
matmul_triton_ch6[(WIDTH // (16*4), WIDTH // 16)](M_t, N_t, P_out_t, WIDTH, TILE=16, GRAN=4)  # warmup
torch.cuda.synchronize()
t0 = time.time()
for _ in range(RUNS):
    matmul_triton_ch6[(WIDTH // (16*4), WIDTH // 16)](M_t, N_t, P_out_t, WIDTH, TILE=16, GRAN=4)
torch.cuda.synchronize()
t_triton = (time.time() - t0) / RUNS * 1000

g_cuda   = gflops(WIDTH, t_best_cuda)
g_triton = gflops(WIDTH, t_triton)

print(f"Best CUDA (tile=16, gran=1x4, prefetch, full unroll): {t_best_cuda:.2f} ms  {g_cuda:.1f} GFLOPS")
print(f"Triton    (TILE=16, GRAN=4):                           {t_triton:.2f} ms  {g_triton:.1f} GFLOPS")
print(f"Winner: {'CUDA' if g_cuda > g_triton else 'Triton'} by {abs(g_cuda - g_triton):.1f} GFLOPS ({abs(g_cuda/g_triton - 1)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Best CUDA", "Triton"], [g_cuda, g_triton],
              color=["#2171b5", "#ef6548"], edgecolor="black", linewidth=0.8, width=0.4)
winner_idx = 0 if g_cuda > g_triton else 1
ax.text(winner_idx, max(g_cuda, g_triton) + 1, "best", ha="center",
        fontsize=11, color="gold", fontweight="bold")
ax.set_ylabel("GFLOPS")
ax.set_title("Best CUDA (Ch. 6) vs Triton")
ax.set_ylim(0, max(g_cuda, g_triton) * 1.2)
for bar, val in zip(bars, [g_cuda, g_triton]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("cuda_vs_triton.png", dpi=150, bbox_inches="tight")
plt.show()